In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

In [3]:
import sqlalchemy as sa
from sqlalchemy.orm import Session
from crmprtd import setup_logging
from pycds import Station, History

save_path = './comparison_forms/'

db_url = "postgresql+psycopg2://tongli1997@/crmp?host=pg01.pcic.uvic.ca,pg02.pcic.uvic.ca&port=5432,5432&target_session_attrs=read-write&passfile=/workspaces/crmprtd/.pgpass"
log_file_path = save_path

engine = sa.create_engine(db_url, echo=False)
session = Session(engine)

session

In [4]:
path = '/workspaces/crmprtd/update_round2/input_tables/'
df = pd.read_excel(path + '20251223-Metadata-AllRequiredChanges_round2.xlsx', header = 1)   # pandas automatically uses openpyxl

# Select rows where ISSUE contains "Attribution" (case-sensitive)
df_attribution = df[df["ISSUE"].str.contains("Attribution", na=False)]

len(df_attribution)

df_attribution =  df_attribution[['ISSUE','Station ID', 'Network ID', 'Network Name', 'Native ID', 'Unique Names', 'Unique Location (String)']].reset_index(drop=True)
df_attribution["Network ID"] = pd.to_numeric(df_attribution["Network ID"], errors="coerce").astype("Int64")
df_attribution["Station ID"] = pd.to_numeric(df_attribution["Station ID"], errors="coerce").astype("Int64")
df_attribution


,ISSUE,Station ID,Network ID,Network Name,Native ID,Unique Names,Unique Location (String)
0,Attribution,2322,12,BC-FWx-> BC-TS,945,TS NAKA CREEK,"115.2144 W, 49.6673 N, Elev. 1051 m -> 126.433..."
1,Attribution,2633,9,BC-Air -> MVRD-AIR,E246240 -> T034,Abbotsford Airport - Walmsley Road,"122.343 W, 49.0235 N, Elev. 65 m"
2,Attribution,12610,9,BC-Air -> MVRD-AIR,New Westminster Sapperton Park -> T046,NA -> New Westminister,"122.894487 W, 49.227045 N, Elev. null m -> 122..."
3,Attribution,12608,9,BC-Air -> MVRD-AIR,Surrey East -> T015,NA -> Surrey East,"122.6942 W, 49.1328 N, Elev. null m -> 122.695..."
4,Attribution,12529,9,BC-Air -> MVRD-AIR,Abbotsford A Columbia Street -> T045,NA -> Abbotsford Airport,"122.3266 W, 49.0215 N, Elev. null m -> 122.326..."
...,...,...,...,...,...,...,...
132,Attribution,2571,14,BC-Snow -> BCH-GSO-HMP,4B15P,Lu Lake,"126.3 W, 54.2 N, Elev. 1310 m"
133,Attribution,2513,14,BC-Snow -> BCH-GSO-HMP,1A02P,McBride (Upper),"120.333 W, 53.3 N, Elev. 1620 m"
134,Attribution,2554,14,BC-Snow -> BCH-GSO-HMP,2F05P,Mission Creek,"118.95 W, 49.95 N, Elev. 1780 m"
135,Attribution,2543,14,BC-Snow -> BCH-GSO-HMP,2A21P,Molson Creek,"118.233 W, 52.2333 N, Elev. 1980 m"


In [5]:
df_attribution = df_attribution[['ISSUE','Station ID', 'Network ID', 'Network Name']]
df_attribution

def split_network_name(name):
    if pd.isna(name):
        return pd.Series([None, None, 0])

    parts = [p.strip() for p in name.split("->")]

    if len(parts) == 2:
        old_name, new_name = parts
        n_names = 2
    else:
        old_name = new_name = parts[0]
        n_names = 1

    return pd.Series([old_name, new_name, n_names])
    

df_attribution[['old_net_name', 'new_net_name', 'n_net_names']] = (
    df_attribution['Network Name'].apply(split_network_name)
)

df_attribution['new_net_name'] = (
    df_attribution['new_net_name']
    .replace(['MVRD-AIR', 'MVRD'], 'MVRD-AQ')
)


df_attribution 

/tmp/ipykernel_60810/811755646.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_attribution[['old_net_name', 'new_net_name', 'n_net_names']] = (
/tmp/ipykernel_60810/811755646.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_attribution[['old_net_name', 'new_net_name', 'n_net_names']] = (


,ISSUE,Station ID,Network ID,Network Name,old_net_name,new_net_name,n_net_names
0,Attribution,2322,12,BC-FWx-> BC-TS,BC-FWx,BC-TS,2
1,Attribution,2633,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AQ,2
2,Attribution,12610,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AQ,2
3,Attribution,12608,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AQ,2
4,Attribution,12529,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AQ,2
...,...,...,...,...,...,...,...
132,Attribution,2571,14,BC-Snow -> BCH-GSO-HMP,BC-Snow,BCH-GSO-HMP,2
133,Attribution,2513,14,BC-Snow -> BCH-GSO-HMP,BC-Snow,BCH-GSO-HMP,2
134,Attribution,2554,14,BC-Snow -> BCH-GSO-HMP,BC-Snow,BCH-GSO-HMP,2
135,Attribution,2543,14,BC-Snow -> BCH-GSO-HMP,BC-Snow,BCH-GSO-HMP,2


### select the netwok_id based on new_net_name, summarize back to the table 

In [7]:
# Create an empty column first
df_attribution['new_network_id'] = None

with engine.connect() as conn:
    for idx, row in df_attribution.iterrows():
        # Query network_id for this row's new_net_name
        res = conn.execute(
            sa.text("""
                SELECT network_id
                FROM meta_network
                WHERE network_display_name = :network_name
            """),
            {"network_name": row['new_net_name']}
        ).mappings().fetchone()

        if res is not None:
            df_attribution.at[idx, 'new_network_id'] = res['network_id']
        else:
            print(f"❌ Network '{row['new_net_name']}' not found for station {row['Station ID']}")

df_attribution

,ISSUE,Station ID,Network ID,Network Name,old_net_name,new_net_name,n_net_names,new_network_id
0,Attribution,2322,12,BC-FWx-> BC-TS,BC-FWx,BC-TS,2,71
1,Attribution,2633,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AQ,2,72
2,Attribution,12610,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AQ,2,72
3,Attribution,12608,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AQ,2,72
4,Attribution,12529,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AQ,2,72
...,...,...,...,...,...,...,...,...
132,Attribution,2571,14,BC-Snow -> BCH-GSO-HMP,BC-Snow,BCH-GSO-HMP,2,5
133,Attribution,2513,14,BC-Snow -> BCH-GSO-HMP,BC-Snow,BCH-GSO-HMP,2,5
134,Attribution,2554,14,BC-Snow -> BCH-GSO-HMP,BC-Snow,BCH-GSO-HMP,2,5
135,Attribution,2543,14,BC-Snow -> BCH-GSO-HMP,BC-Snow,BCH-GSO-HMP,2,5


In [8]:
df_attribution['new_network'] = df_attribution['new_network_id']> 50
df_attri_new_net = df_attribution[df_attribution['new_network'] == True]
df_attri_old_net = df_attribution[df_attribution['new_network'] == False]
df_attri_old_net = df_attri_old_net.drop(columns=['n_net_names'])
df_attri_old_net = df_attri_old_net.rename(columns={'Network ID': 'old_network_id'})

df_attri_new_net = df_attri_new_net.reset_index(drop=True)
df_attri_new_net

,ISSUE,Station ID,Network ID,Network Name,old_net_name,new_net_name,n_net_names,new_network_id,new_network
0,Attribution,2322,12,BC-FWx-> BC-TS,BC-FWx,BC-TS,2,71,True
1,Attribution,2633,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AQ,2,72,True
2,Attribution,12610,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AQ,2,72,True
3,Attribution,12608,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AQ,2,72,True
4,Attribution,12529,9,BC-Air -> MVRD-AIR,BC-Air,MVRD-AQ,2,72,True
...,...,...,...,...,...,...,...,...,...
80,"Location, Attribution",12453,12,BC-FWx-> BC-TS,BC-FWx,BC-TS,2,71,True
81,"Name, Location, Attribution",12458,12,BC-FWx-> BC-TS,BC-FWx,BC-TS,2,71,True
82,"Name, Location, Attribution",12526,12,BC-FWx-> BC-TS,BC-FWx,BC-TS,2,71,True
83,"Name, Location, Attribution",12957,12,BC-FWx-> BC-TS,BC-FWx,BC-TS,2,71,True


In [ ]:
# df_attri_old_net

# df_attri_old_net.to_csv(
#     "./output_tables/Network_attributed_to_exist_network_info.csv",
#     index=False
# )


In [10]:
from sqlalchemy import text

SQL_STATION_VARS = text("""
SELECT DISTINCT
        v.vars_id,
        v.unit,
        v.precision,
        v.standard_name,
        v.cell_method,
        v.long_description,
        v.net_var_name,
        v.display_name,
        v.short_name
FROM obs_raw o
JOIN meta_history h ON o.history_id = h.history_id
JOIN meta_station s ON h.station_id = s.station_id
JOIN meta_vars v ON o.vars_id = v.vars_id
WHERE s.station_id = ANY(:station_ids)
ORDER BY v.vars_id
""")

station_ids = (
    df_attri_old_net["Station ID"]
    .dropna()
    .astype(int)
    .tolist()
)        
with engine.connect() as conn:
    old_stations_vars = pd.read_sql(SQL_STATION_VARS, conn, params={"station_ids": station_ids})

In [ ]:
new_network_ids = (
    df_attri_old_net["new_network_id"]
    .dropna()
    .astype(int)
    .unique()
    .tolist()
)

SQL_VARS_NEW = text("""
SELECT DISTINCT
       v.vars_id,
       v.unit,
       v.precision,
       v.standard_name,
       v.cell_method,
       v.long_description,
       v.net_var_name,
       v.display_name,
       v.short_name,
       v.network_id
FROM meta_vars v
WHERE v.network_id = ANY(:network_ids)
""")

new_net_vars = pd.read_sql(
    SQL_VARS_NEW,
    engine,
    params={"network_ids": new_network_ids}
)

new_net_vars

,vars_id,unit,precision,standard_name,cell_method,long_description,net_var_name,display_name,short_name,network_id
0,836,m3/m3,None,liquid_water_content_of_soil_layer,time: mean (interval: 1 hour),Liquid water content of soil - 30 cm,Water_Content_m3_m3_30_cm,Soil Liquid Water Content - 30 cm,None,11
1,672,W m-2,None,downward_heatflux_in_soil,time: mean,soil heat flux ( 1 of 2),HFT3_1_Avg,Soil Heatflux 1,soil_heatflux_1,5
2,678,W m-2,None,upwelling_shortwave_flux_in_air,time: mean,upwelling shortwave radiation,K_up_Avg,Upwelling Shortwave Radiation,upwelling_shortwave_radiation_upwelling_mean,5
3,655,mm,None,lwe_thickness_of_precipitation_amount,t: sum over days interval: irregular,Standpipe total precipitation since last gauge...,Precip,Precipitation (Cumulative),lwe_thickness_of_precipitation_accumulated,5
4,571,celsius,None,air_temperature,t: mean within days t: mean within months t: m...,Climatological mean of monthly mean of mean da...,T_mean_Climatology,Temperature Climatology (Mean),air_temperaturet: mean within days t: mean wit...,5
...,...,...,...,...,...,...,...,...,...,...
68,803,%,None,relative_humidity,time: mean,None,avg_rel_hum_pst1hr,Relative Humidity,rh,15
69,791,km/h,None,wind_speed,time: point,None,WSPD_SCLR,Wind Speed (Point),wind_speed_point,15
70,509,W m-2,None,surface_downwelling_shortwave_flux,time: mean,None,SolarRadiationWm,Incoming Shortwave,surface_downwelling_shortwave_flux_mean,11
71,538,m s-1,None,wind_speed,time: minimum,None,Wind_n,Wind Speed (Min.),wind_speed_minimum,11


In [ ]:
# output_path = "./output_tables/Old_new_network_info_vars.xlsx"

# with pd.ExcelWriter(output_path, engine="xlsxwriter") as writer:
#     df_attri_old_net.to_excel(
#         writer,
#         sheet_name="Network_station_info",
#         index=False
#     )
#     old_stations_vars.to_excel(
#         writer,
#         sheet_name="old_station_vars",
#         index=False
#     )
#     new_net_vars.to_excel(
#         writer,
#         sheet_name="new_network_vars",
#         index=False
#     )

# print(f"📁 Saved to {output_path}")

📁 Saved to ./output_tables/Old_new_network_info_vars.xlsx


In [27]:
from sqlalchemy import text
import pandas as pd
from sqlalchemy.engine import Engine
from typing import Optional, List

def compare_network_vars(
    engine: Engine,
    df_attri_old_net: pd.DataFrame,
    old_network_id: int,
    new_network_id: int,
    station_ids: Optional[List[int]] = None
) -> pd.DataFrame:
    """
    Compare variables used by stations in an old network to variables in a new network.

    Parameters
    ----------
    engine : Engine
        SQLAlchemy engine connected to the database.
    df_attri_old_net : pd.DataFrame
        DataFrame containing at least 'Station ID' and 'old_network_id'.
    old_network_id : int
        old_network_id of the old network whose stations we want to check.
    new_network_id : int
        old_network_id of the target network to compare variables against.
    station_ids : Optional[List[int]]
        List of station IDs to check. If None, all stations in the old network are used.

    Returns
    -------
    pd.DataFrame
        DataFrame showing old station variables, matching new network vars, 
        and a boolean column 'exists_in_new_net'.
    """

    # Get station IDs if not provided
    if station_ids is None:
        station_ids = df_attri_old_net.loc[
            df_attri_old_net['old_network_id'] == old_network_id, 'Station ID'
        ].tolist()

    if not station_ids:
        print("⚠️ No stations found for the old network.")
        return pd.DataFrame()

    # --- Load variables used by the old stations ---
    SQL_STATION_VARS = text("""
    SELECT DISTINCT
           v.vars_id,
           v.unit,
           v.precision,
           v.standard_name,
           v.cell_method,
           v.long_description,
           v.net_var_name,
           v.display_name,
           v.short_name
    FROM obs_raw o
    JOIN meta_history h ON o.history_id = h.history_id
    JOIN meta_station s ON h.station_id = s.station_id
    JOIN meta_vars v ON o.vars_id = v.vars_id
    WHERE s.station_id = ANY(:station_ids)
    ORDER BY v.vars_id
    """)

    with engine.connect() as conn:
        df_vars_old_stations = pd.read_sql(SQL_STATION_VARS, conn, params={"station_ids": station_ids})

    if df_vars_old_stations.empty:
        print("⚠️ No variables found for the old stations.")
        return pd.DataFrame()

    # --- Load variables in the new network ---
    SQL_VARS_NEW = text("""
    SELECT DISTINCT
           v.vars_id,
           v.unit,
           v.precision,
           v.standard_name,
           v.cell_method,
           v.long_description,
           v.net_var_name,
           v.display_name,
           v.short_name
    FROM meta_vars v
    JOIN meta_network n ON v.network_id = n.network_id
    WHERE n.network_id = :network_id
    ORDER BY v.vars_id

    """)

    new_net_vars = pd.read_sql(SQL_VARS_NEW, engine, params={"network_id": new_network_id})

    # --- Compare variables ---
    VAR_MATCH_COLS = [
        "unit",
        "precision",
        "standard_name",
        "cell_method",
        "long_description",
        "net_var_name",
        "display_name",
        "short_name",
    ]

    df_compare = df_vars_old_stations.merge(
        new_net_vars[VAR_MATCH_COLS + ["vars_id"]],
        on=VAR_MATCH_COLS,
        how="left",
        suffixes=("", "_new_net"),
        indicator=True
    )

    # Mark existence in new network
    df_compare["exists_in_new_net"] = df_compare["_merge"] == "both"
    df_compare["vars_id_new_net"] = df_compare["vars_id_new_net"]

    return df_compare, df_vars_old_stations, new_net_vars

#### Matching between different pairs, and save the file

In [ ]:
# import pandas as pd

# network_pairs = [
#     (14, 5, "14_to_5"),
#     (12, 15, "12_to_15"),
#     (12, 11, "12_to_11")
# ]

# output_file = "./output_tables/Reattribution_network_info_3pairs_vars_match.xlsx"

# with pd.ExcelWriter(output_file, engine="xlsxwriter") as writer:

#     # --- 1️⃣ Write df_attri_old_net as the first sheet ---
#     df_attri_old_net.to_excel(
#         writer,
#         sheet_name="Network_station_info",
#         index=False
#     )

#     # Auto-adjust column width for the first sheet
#     worksheet = writer.sheets["Network_station_info"]
#     for i, col in enumerate(df_attri_old_net.columns):
#         max_len = max(df_attri_old_net[col].astype(str).map(len).max(), len(str(col)))
#         worksheet.set_column(i, i, max_len + 2)

#     # --- 2️⃣ Write the comparison sheets ---
#     for old_net, new_net, sheet_name in network_pairs:

#         df_compare_pair, old_station_var, new_net_vars = compare_network_vars(
#             engine=engine,
#             df_attri_old_net=df_attri_old_net,
#             old_network_id=old_net,
#             new_network_id=new_net
#         )

#         old_label = pd.DataFrame([["Old Network, Station Vars"] + [""]*(old_station_var.shape[1]-1)], columns=old_station_var.columns)
#         new_label = pd.DataFrame([["New Network Vars"] + [""]*(new_net_vars.shape[1]-1)], columns=new_net_vars.columns)
#         blank_row = pd.DataFrame([[""]*old_station_var.shape[1]], columns=old_station_var.columns)

#         df_to_save = pd.concat([
#             old_label,
#             blank_row,
#             old_station_var,
#             pd.DataFrame([[""]*old_station_var.shape[1]]*2, columns=old_station_var.columns),
#             new_label,
#             blank_row,
#             new_net_vars
#         ], ignore_index=True)

#         df_to_save.to_excel(writer, sheet_name=sheet_name, index=False)

#         # Auto-adjust column width
#         worksheet = writer.sheets[sheet_name]
#         for i, col in enumerate(df_to_save.columns):
#             max_len = max(df_to_save[col].astype(str).map(len).max(), len(str(col)))
#             worksheet.set_column(i, i, max_len + 2)

# print(f"✅ Excel file saved with df_attri_old_net as first sheet and comparison sheets unfolded: {output_file}")

✅ Excel file saved with df_attri_old_net as first sheet and comparison sheets unfolded: ./output_tables/Reattribution_network_info_3pairs_vars_match.xlsx


In [22]:
df_compare_pair1, old_station_var, new_net_var = compare_network_vars(
    engine=engine,
    df_attri_old_net=df_attri_old_net,
    old_network_id=14,
    new_network_id=5
)
df_compare_pair1

,vars_id,unit,precision,standard_name,cell_method,long_description,net_var_name,display_name,short_name,vars_id_new_net,_merge,exists_in_new_net
0,511,celsius,None,air_temperature,time: minimum,Minimum daily temperature,MIN_TEMP,Temperature (Min.),air_temperature_minimum,463.0,both,True
1,512,celsius,None,air_temperature,time: maximum,Maximum daily temperature,MAX_TEMP,Temperature (Max.),air_temperature_maximum,464.0,both,True
2,513,mm,None,lwe_thickness_of_precipitation_amount,time: sum,Daily precipitation,ONE_DAY_PRECIPITATION,Precipitation Amount,lwe_thickness_of_precipitation_amount_sum,465.0,both,True
3,515,cm,None,thickness_of_snowfall_amount,time: sum,Daily snow accumulation,ONE_DAY_SNOW,Snowfall Amount,thickness_of_snowfall_amount_sum,467.0,both,True
4,516,cm,None,surface_snow_thickness,time: point,Amount of snow on the ground,SNOW_ON_THE_GROUND,Surface Snow Depth (Point),surface_snow_thickness_point,468.0,both,True
5,597,celsius,None,air_temperature,t: maximum within days t: mean within months t...,Climatological mean of monthly mean maximum da...,Tx_Climatology,Temperature Climatology (Max.),air_temperaturet: maximum within days t: mean ...,569.0,both,True
6,598,celsius,None,air_temperature,t: minimum within days t: mean within months t...,Climatological mean of monthly mean minimum da...,Tn_Climatology,Temperature Climatology (Min.),air_temperaturet: minimum within days t: mean ...,570.0,both,True
7,599,celsius,None,air_temperature,t: mean within days t: mean within months t: m...,Climatological mean of monthly mean of mean da...,T_mean_Climatology,Temperature Climatology (Mean),air_temperaturet: mean within days t: mean wit...,571.0,both,True
8,600,mm,None,lwe_thickness_of_precipitation_amount,t: sum within months t: mean over years,Climatological mean of monthly total precipita...,Precip_Climatology,Precipitation Climatology,lwe_thickness_of_precipitation_amountt: sum wi...,572.0,both,True
9,660,mm,None,liquid_water_content_of_surface_snow,time: point,Snow water equivalent of snow on the ground,SWE,Snow Water Equivalent,swe,NaN,left_only,False


In [26]:


df_compare_pair1['vars_id_new_net'] = pd.to_numeric(df_compare_pair1['vars_id_new_net'], errors="coerce").astype("Int64")

df_compare_pair1['vars_id_new_net']

0      463
1      464
2      465
3      467
4      468
5      569
6      570
7      571
8      572
9     <NA>
10    <NA>
11    <NA>
12    <NA>
13    <NA>
14    <NA>
15    <NA>
16    <NA>
Name: vars_id_new_net, dtype: Int64

In [21]:
df_compare_pair2, old_station_var, new_net_var = compare_network_vars(
    engine=engine,
    df_attri_old_net=df_attri_old_net,
    old_network_id=12,
    new_network_id=15
)
df_compare_pair2

(    vars_id     unit precision                          standard_name  \
 0       496       mm      None  lwe_thickness_of_precipitation_amount   
 1       497  celsius      None                        air_temperature   
 2       498        %      None                      relative_humidity   
 3       499     km/h      None                             wind_speed   
 4       500   degree      None                    wind_from_direction   
 5       711  celsius      None                        air_temperature   
 6       712       mm      None           thickness_of_rainfall_amount   
 7       713        %      None                      relative_humidity   
 8       714     km/h      None                             wind_speed   
 9       715  degrees      None                    wind_from_direction   
 10      718       mm      None           thickness_of_rainfall_amount   
 11      719  celsius      None                  dew_point_temperature   
 12      736       cm      None       

In [ ]:

660 -> 659
    660	mm	None	liquid_water_content_of_surface_snow	time: point	Snow water equivalent of snow on the ground	SWE	Snow Water Equivalent	swe	

    659	mm	None	liquid_water_content_of_surface_snow	time: point	Snow water equivalent of snow on the ground	Snow_WE	Snow Water Equivalent	swe

703 -> 467 / 704 -> 467?
    703	mm	None	lwe_thickness_of_surface_snow_amount	time: sum	None	snw_dpth_wtr_equiv	Snow Water Equivalent	swe	

    467	cm	None	thickness_of_snowfall_amount	time: sum	Daily snow accumulation	ONE_DAY_SNOW	Snowfall Amount	thickness_of_snowfall_amount_sum

    The difference between 703 and 704? 

    704	cm	None	surface_snow_thickness	time: sum	None	snw_dpth	Snow Depth	sd	



710 -> 465 or 466 or 655?
    710	mm	None	lwe_thickness_of_precipitation_amount	time: sum (interval: 24 hour)	None	pcpn_amt_pst24hrs	Precipitation 24hr	ppt_24hr	

    465	mm	None	lwe_thickness_of_precipitation_amount	time: sum	Daily precipitation	ONE_DAY_PRECIPITATION	Precipitation Amount	lwe_thickness_of_precipitation_amount_sum

    466	mm	None	thickness_of_rainfall_amount	time: sum	Daily rainfall	ONE_DAY_RAIN	Rainfall Amount	thickness_of_rainfall_amount_sum

    655	mm	None	lwe_thickness_of_precipitation_amount	t: sum over days interval: irregular	Standpipe total precipitation since last gauge...	Precip	Precipitation (Cumulative)	lwe_thickness_of_precipitation_accumulated

705 Monthly?
    705	mm	None	lwe_thickness_of_precipitation_amount	time: sum	None	cum_pcpn_amt	Cumulative Precipitation	ppt_tot

    572	mm	None	lwe_thickness_of_precipitation_amount	t: sum within months t: mean over years	Climatological mean of monthly total precipita...	Precip_Climatology	Precipitation Climatology	lwe_thickness_of_precipitation_amountt: sum wi...



706	mm	None	lwe_thickness_of_precipitation_amount	time: sum (interval: 1 hour)	None	pcpn_amt_pst1hr	Precipitation 1hr	ppt	

    no correponding variables in network 5


station variables
10	701	celsius	None	air_temperature	time: point	None	air_temp_1	Temperature (Mean)	T	
11	702	celsius	None	air_temperature	time: point	None	air_temp_2	Temperature (Mean)	T	
	


,vars_id,unit,precision,standard_name,cell_method,long_description,net_var_name,display_name,short_name
0,676,W m-2,None,upwelling_longwave_flux_in_air,time: mean,upwelling longwave radiation,L_up_Avg,Upwelling Longwave Radiation,upwelling_longwave_upwelling_mean
1,815,degree,None,wind_from_direction,time: mean,Wind Direction,WindDirection,Wind Direction (Mean),wind_from_direction_mean
2,817,millibar,None,air_pressure,time: point,Atmospheric pressure,BarometricPressure,Air Pressure (Point),air_pressure_point
3,681,W m-2,None,surface_net_downward_radiative_flux,time: mean,Net Radiation,NetRad,Net Radiation,net_radiation
4,468,cm,None,surface_snow_thickness,time: point,Amount of snow on the ground,SNOW_ON_THE_GROUND,Surface Snow Depth (Point),surface_snow_thickness_point
5,571,celsius,None,air_temperature,t: mean within days t: mean within months t: m...,Climatological mean of monthly mean of mean da...,T_mean_Climatology,Temperature Climatology (Mean),air_temperaturet: mean within days t: mean wit...
6,657,celsius,None,air_temperature,time: point,Hourly air temperature instantaneous,AirTemp,Temperature (Point),air_temperature_point
7,673,W m-2,None,downward_heatflux_in_soil,time: mean,soil heat flux ( 2 of 2),HFT3_2_Avg,Soil Heatflux 2,soil_heatflux_2
8,672,W m-2,None,downward_heatflux_in_soil,time: mean,soil heat flux ( 1 of 2),HFT3_1_Avg,Soil Heatflux 1,soil_heatflux_1
9,675,W m-2,None,downwelling_longwave_flux_in_air,time: mean,downwelling longwave radiation (temperature co...,L_down_corr_Avg,Downwelling Longwave Radiation - temperature c...,downwelling_longwave_corr_radiation


### Matching of pair 2 (12 -> 15)

In [24]:
df_compare_pair2 = compare_network_vars(
    engine=engine,
    df_attri_old_net=df_attri_old_net,
    old_network_id=12,
    new_network_id=15
)
df_compare_pair2

,vars_id,unit,precision,standard_name,cell_method,long_description,net_var_name,display_name,short_name,vars_id_new_net,_merge,exists_in_new_net
0,496,mm,None,lwe_thickness_of_precipitation_amount,time: sum,depth of water-equivalent rain,precipitation,Precipitation Amount,lwe_thickness_of_precipitation_amount_sum,NaN,left_only,False
1,497,celsius,None,air_temperature,time: point,None,temperature,Temperature (Point),air_temperature_point,NaN,left_only,False
2,498,%,None,relative_humidity,time: mean,None,relative_humidity,Relative Humidity (Mean),relative_humidity_mean,NaN,left_only,False
3,499,km/h,None,wind_speed,time: mean,None,wind_speed,Wind Speed (Mean),wind_speed_mean,NaN,left_only,False
4,500,degree,None,wind_from_direction,time: mean,None,wind_direction,Wind Direction (Mean),wind_from_direction_mean,NaN,left_only,False
5,711,celsius,None,air_temperature,time: point,None,air_temp,Temperature,T,NaN,left_only,False
6,712,mm,None,thickness_of_rainfall_amount,time: sum (interval: 1 hour),None,rnfl_amt_pst1hr,Rainfall 1hr,1hr rain,NaN,left_only,False
7,713,%,None,relative_humidity,time: point,None,rel_hum,Relative Humidity,RH,NaN,left_only,False
8,714,km/h,None,wind_speed,time: mean,None,avg_wnd_spd_10m_pst10mts,Wind Speed,Wind Speed,NaN,left_only,False
9,715,degrees,None,wind_from_direction,time: mean,None,avg_wnd_dir_10m_pst10mts,Wind Direction,wdir,NaN,left_only,False


In [ ]:
0	496	mm	None	lwe_thickness_of_precipitation_amount	time: sum	depth of water-equivalent rain	precipitation	Precipitation Amount	lwe_thickness_of_precipitation_amount_sum	
	604,mm,,lwe_thickness_of_precipitation_amount,t: sum within months t: mean over years,Climatological mean of monthly total precipitation,Precip_Climatology,Precipitation Climatology,lwe_thickness_of_precipitation_amountt: sum within months t: mean over years

6	712	mm	None	thickness_of_rainfall_amount	time: sum (interval: 1 hour)	None	rnfl_amt_pst1hr	Rainfall 1hr	1hr rain
        793,mm,,lwe_thickness_of_precipitation_amount,time: sum,Hourly precipitation,PRECIP_TOTAL,Precipitation Amount,lwe_thickness_of_precipitation_amount_sum

10	718	mm	None	thickness_of_rainfall_amount	time: sum (interval: 24 hour)	None	rnfl_amt_pst24hrs	Rainfall 24hr	rain_24hr



1	497	celsius	None	air_temperature	time: point	None	temperature	Temperature (Point)	air_temperature_point	
5	711	celsius	None	air_temperature	time: point	None	air_temp	Temperature	T	

        792,celsius,,air_temperature,time: mean,Present temperature,TEMP_MEAN,Temperature (Mean),air_temperature_mean
        800,celsius,,air_temperature,time: mean,,avg_air_temp_pst1hr,Temperature (Mean),T



2	498	%	None	relative_humidity	time: mean	None	relative_humidity	Relative Humidity (Mean)	relative_humidity_mean	
        803,%,,relative_humidity,time: mean,,avg_rel_hum_pst1hr,Relative Humidity,rh


7	713	%	None	relative_humidity	time: point	None	rel_hum	Relative Humidity	RH	
        794,%,,relative_humidity,time: point,Relative humidity,HUMIDITY,Relative Humidity (Point),relative_humidity_point



3	499	km/h	None	wind_speed	time: mean	None	wind_speed	Wind Speed (Mean)	wind_speed_mean	

8	714	km/h	None	wind_speed	time: mean	None	avg_wnd_spd_10m_pst10mts	Wind Speed	Wind Speed	
        801,km/h,,wind_speed,time: mean,,avg_wnd_spd_sclr_10m_pst1hr,Wind Speed,wind
        802,km/h,,wind_speed,time: mean,,avg_wnd_spd_10m_pst1hr,Wind Speed Scalar,wind
        791,km/h,,wind_speed,time: point,,WSPD_SCLR,Wind Speed (Point),wind_speed_point



4	500	degree	None	wind_from_direction	time: mean	None	wind_direction	Wind Direction (Mean)	wind_from_direction_mean	


9	715	degrees	None	wind_from_direction	time: mean	None	avg_wnd_dir_10m_pst10mts	Wind Direction	wdir	

        796,degrees,,wind_from_direction,time: point,,WDIR_VECT,Wind Direction (Point),wind_from_direction_point
        797,degrees,,wind_from_direction,time: mean,,avg_wnd_dir_10m_pst1hr,Wind Direction,wdir




	

New:
11	719	celsius	None	dew_point_temperature	time: point	None	dwpt_temp	Dew Point Temperature	Td	


12	736	cm	None	surface_snow_thickness	time: point	None	snw_dpth	Snow Depth	sd	




In [ ]:



601,celsius,,air_temperature,t: maximum within days t: mean within months t: mean over years,Climatological mean of monthly mean maximum daily temperature,Tx_Climatology,Temperature Climatology (Max.),air_temperaturet: maximum within days t: mean within months t: mean over years



799,degrees,,wind_from_direction_std_deviation,time: standard_deviation (interval: 1 hour),,std_dev_wnd_dir_10m_pst1hr,Wind Direction Standard Deviation,wind_var


804,hPa,,water_vapor_partial_pressure_in_air,time: point,,vpr_pres,Vapour Pressure,vp



798,degrees,,wind_from_direction_unit_vector,time: mean,,avg_unit_vtr_wnd_dir_10m_pst1hr,Wind Direction Unit Vector,wdir

602,celsius,,air_temperature,t: minimum within days t: mean within months t: mean over years,Climatological mean of monthly mean minimum daily temperature,Tn_Climatology,Temperature Climatology (Min.),air_temperaturet: minimum within days t: mean within months t: mean over years

603,celsius,,air_temperature,t: mean within days t: mean within months t: mean over years,Climatological mean of monthly mean of mean daily temperature,T_mean_Climatology,Temperature Climatology (Mean),air_temperaturet: mean within days t: mean within months t: mean over years


795,millibar,,air_pressure,time: point,Atmospheric pressure,BAR_PRESS,Air Pressure (Point),air_pressure_point


### Matching of pair 3 (12 -> 11)


In [25]:
df_compare_pair3 = compare_network_vars(
    engine=engine,
    df_attri_old_net=df_attri_old_net,
    old_network_id=12,
    new_network_id=11
)
df_compare_pair3

,vars_id,unit,precision,standard_name,cell_method,long_description,net_var_name,display_name,short_name,vars_id_new_net,_merge,exists_in_new_net
0,496,mm,None,lwe_thickness_of_precipitation_amount,time: sum,depth of water-equivalent rain,precipitation,Precipitation Amount,lwe_thickness_of_precipitation_amount_sum,NaN,left_only,False
1,497,celsius,None,air_temperature,time: point,None,temperature,Temperature (Point),air_temperature_point,NaN,left_only,False
2,498,%,None,relative_humidity,time: mean,None,relative_humidity,Relative Humidity (Mean),relative_humidity_mean,NaN,left_only,False
3,499,km/h,None,wind_speed,time: mean,None,wind_speed,Wind Speed (Mean),wind_speed_mean,NaN,left_only,False
4,500,degree,None,wind_from_direction,time: mean,None,wind_direction,Wind Direction (Mean),wind_from_direction_mean,NaN,left_only,False
5,711,celsius,None,air_temperature,time: point,None,air_temp,Temperature,T,NaN,left_only,False
6,712,mm,None,thickness_of_rainfall_amount,time: sum (interval: 1 hour),None,rnfl_amt_pst1hr,Rainfall 1hr,1hr rain,NaN,left_only,False
7,713,%,None,relative_humidity,time: point,None,rel_hum,Relative Humidity,RH,NaN,left_only,False
8,714,km/h,None,wind_speed,time: mean,None,avg_wnd_spd_10m_pst10mts,Wind Speed,Wind Speed,NaN,left_only,False
9,715,degrees,None,wind_from_direction,time: mean,None,avg_wnd_dir_10m_pst10mts,Wind Direction,wdir,NaN,left_only,False


In [ ]:
0	496	mm	None	lwe_thickness_of_precipitation_amount	time: sum	depth of water-equivalent rain	precipitation	Precipitation Amount	lwe_thickness_of_precipitation_amount_sum

10	718	mm	None	thickness_of_rainfall_amount	time: sum (interval: 24 hour)	None	rnfl_amt_pst24hrs	Rainfall 24hr	rain_24hr

6	712	mm	None	thickness_of_rainfall_amount	time: sum (interval: 1 hour)	None	rnfl_amt_pst1hr	Rainfall 1hr	1hr rain


1	497	celsius	None	air_temperature	time: point	None	temperature	Temperature (Point)	air_temperature_point
    503,celsius,,air_temperature,time: point,,TempC,Temperature (Point),air_temperature_point

5	711	celsius	None	air_temperature	time: point	None	air_temp	Temperature	T
    529,celsius,,air_temperature,time: mean,,Tm,Temperature (Mean),air_temperature_mean



2	498	%	None	relative_humidity	time: mean	None	relative_humidity	Relative Humidity (Mean)	relative_humidity_mean
    504,%,,relative_humidity,time: mean,,RH,Relative Humidity (Mean),relative_humidity_mean

7	713	%	None	relative_humidity	time: point	None	rel_hum	Relative Humidity	RH


3	499	km/h	None	wind_speed	time: mean	None	wind_speed	Wind Speed (Mean)	wind_speed_mean
    506,m s-1,,wind_speed,time: mean,,WindSpeedms,Wind Speed (Mean),wind_speed_mean

8	714	km/h	None	wind_speed	time: mean	None	avg_wnd_spd_10m_pst10mts	Wind Speed	Wind Speed


4	500	degree	None	wind_from_direction	time: mean	None	wind_direction	Wind Direction (Mean)	wind_from_direction_mean
    508,degree,,wind_from_direction,time: mean,,WindDirection,Wind Direction (Mean),wind_from_direction_mean

9	715	degrees	None	wind_from_direction	time: mean	None	avg_wnd_dir_10m_pst10mts	Wind Direction	wdir


11	719	celsius	None	dew_point_temperature	time: point	None	dwpt_temp	Dew Point Temperature	Td
    505,celsius,,dew_point_temperature,time: mean,,DewPtC,Dew Point Temperature (Mean),dew_point_temperature_mean


12	736	cm	None	surface_snow_thickness	time: point	None	snw_dpth	Snow Depth	sd
    832,cm,,surface_snow_depth,time: point,Surface snow depth,Snow_Depth,Snow Depth,Snow Depth


In [ ]:
538,m s-1,,wind_speed,time: minimum,,Wind_n,Wind Speed (Min.),wind_speed_minimum
502,millibar,,air_pressure,time: point,,Pressurembar,Air Pressure (Point),air_pressure_point
501,mm,,lwe_thickness_of_precipitation_amount,time: sum,depth of water-equivalent rain,Rainmm,Precipitation Amount,lwe_thickness_of_precipitation_amount_sum
534,%,,relative_humidity,time: maximum,,RHx,Relative Humidity (Max.),relative_humidity_maximum
586,celsius,,air_temperature,t: minimum within days t: mean within months t: mean over years,Climatological mean of monthly mean minimum daily temperature,Tn_Climatology,Temperature Climatology (Min.),air_temperaturet: minimum within days t: mean within months t: mean over years
835,m3/m3,,liquid_water_content_of_soil_layer,time: mean (interval: 1 hour),Liquid water content of soil - 15 cm,Water_Content_m3_m3_15_cm,Soil Liquid Water Content - 15 cm,
834,m3/m3,,liquid_water_content_of_soil_layer,time: mean (interval: 1 hour),Liquid water content of soil - 5 cm,Water_Content_m3_m3_5_cm,Soil Liquid Water Content - 5cm,
530,celsius,,air_temperature,time: maximum,,Tx,Temperature (Max.),air_temperature_maximum
588,mm,,lwe_thickness_of_precipitation_amount,t: sum within months t: mean over years,Climatological mean of monthly total precipitation,Precip_Climatology,Precipitation Climatology,lwe_thickness_of_precipitation_amountt: sum within months t: mean over years
507,m s-1,,wind_speed_of_gust,time: maximum,,GustSpeedms,Wind Gust (Max.),wind_speed_of_gust_maximum
535,%,,relative_humidity,time: minimum,,RHn,Relative Humidity (Min.),relative_humidity_minimum
833,celsius,,soil_temperature,time: mean (interval: 1 hour),Soil temperature,Soil_Temp,Soil Temperature,Soil Temperature
682,%,,volume_fraction_of_condensed_water_in_soil ,time:point,Soil Moisture at 20cm depth,Wetness_20cm,Wetness_20cm,Wetness_20cm
509,W m-2,,surface_downwelling_shortwave_flux,time: mean,,SolarRadiationWm,Incoming Shortwave,surface_downwelling_shortwave_flux_mean
587,celsius,,air_temperature,t: mean within days t: mean within months t: mean over years,Climatological mean of monthly mean of mean daily temperature,T_mean_Climatology,Temperature Climatology (Mean),air_temperaturet: mean within days t: mean within months t: mean over years
510,%,,volume_fraction_of_condensed_water_in_soil,time: point,,Wetness,Soil Moisture,volume_fraction_of_condensed_water_in_soil_point
836,m3/m3,,liquid_water_content_of_soil_layer,time: mean (interval: 1 hour),Liquid water content of soil - 30 cm,Water_Content_m3_m3_30_cm,Soil Liquid Water Content - 30 cm,
585,celsius,,air_temperature,t: maximum within days t: mean within months t: mean over years,Climatological mean of monthly mean maximum daily temperature,Tx_Climatology,Temperature Climatology (Max.),air_temperaturet: maximum within days t: mean within months t: mean over years
531,celsius,,air_temperature,time: minimum,,Tn,Temperature (Min.),air_temperature_minimum


In [ ]:
old_net_vars
 
new_net_vars

,vars_id,unit,precision,standard_name,cell_method,long_description,net_var_name,display_name,short_name
0,513,mm,None,lwe_thickness_of_precipitation_amount,time: sum,Daily precipitation,ONE_DAY_PRECIPITATION,Precipitation Amount,lwe_thickness_of_precipitation_amount_sum
1,705,mm,None,lwe_thickness_of_precipitation_amount,time: sum,None,cum_pcpn_amt,Cumulative Precipitation,ppt_tot
2,710,mm,None,lwe_thickness_of_precipitation_amount,time: sum (interval: 24 hour),None,pcpn_amt_pst24hrs,Precipitation 24hr,ppt_24hr
3,702,celsius,None,air_temperature,time: point,None,air_temp_2,Temperature (Mean),T
4,660,mm,None,liquid_water_content_of_surface_snow,time: point,Snow water equivalent of snow on the ground,SWE,Snow Water Equivalent,swe
5,514,mm,None,thickness_of_rainfall_amount,time: sum,Daily rainfall,ONE_DAY_RAIN,Rainfall Amount,thickness_of_rainfall_amount_sum
6,515,cm,None,thickness_of_snowfall_amount,time: sum,Daily snow accumulation,ONE_DAY_SNOW,Snowfall Amount,thickness_of_snowfall_amount_sum
7,706,mm,None,lwe_thickness_of_precipitation_amount,time: sum (interval: 1 hour),None,pcpn_amt_pst1hr,Precipitation 1hr,ppt
8,600,mm,None,lwe_thickness_of_precipitation_amount,t: sum within months t: mean over years,Climatological mean of monthly total precipita...,Precip_Climatology,Precipitation Climatology,lwe_thickness_of_precipitation_amountt: sum wi...
9,512,celsius,None,air_temperature,time: maximum,Maximum daily temperature,MAX_TEMP,Temperature (Max.),air_temperature_maximum
